# Liquidation Strategy Inspection

This inspection walkthrough is a low-level diagnostic notebook for the V1 liquidation strategy engine with the synthetic sample data. It intentionally uses direct loaders and direct engine calls so reviewers can inspect intermediate liquidation inputs.

This notebook is not the application workflow. The application-aligned walkthrough is [`liquidation_strategy_walkthrough.ipynb`](liquidation_strategy_walkthrough.ipynb), which calls the same service layer as Streamlit.

The goal here is to make the liquidation process easy to inspect: scenario inputs, cash-buffer preservation, eligibility, gross liquidation allocations, post-haircut cash raised, dilution, shortfall, and asset-group allocations.

## Imports And Setup

This section imports the shared setup helper, existing project loaders, domain labels, and liquidation strategy engine. It also defines small display helpers for readable tables. These helpers are local diagnostic aids and are not business logic that the application will call.

In [1]:
from decimal import Decimal

import pandas as pd
from _notebook_setup import SAMPLE_DATA_DIR, configure_display

from lmt_calibration.domain import AssetGroup
from lmt_calibration.engines import StressedLiquidationPosition, calculate_liquidation_strategy
from lmt_calibration.loaders import (
    load_funds_csv,
    load_investor_classes_csv,
    load_liquidation_strategies_json,
    load_liquidity_stresses_json,
    load_lmt_parameters_csv,
    load_market_stresses_csv,
    load_positions_csv,
    load_redemption_scenarios_csv,
    load_scenario_definitions_csv,
)

configure_display()


def money(value: Decimal | int | float | None) -> str:
    if value is None:
        return ""
    return f"{Decimal(value):,.2f}"


def rate(value: Decimal | int | float | None) -> str:
    if value is None:
        return ""
    return f"{Decimal(value) * Decimal('100'):.2f}%"


def days(value: int | None) -> str:
    if value is None:
        return ""
    return str(value)

## Sample Data Loaded

This section loads every sample dataset through the existing loaders. This matters because the inspection should use the same validated domain objects that the application code uses, rather than bypassing validation with ad hoc CSV reads.

Interpret the output as a quick inventory of the sample inputs available for the liquidation review.

In [2]:
funds = load_funds_csv(SAMPLE_DATA_DIR / "funds.csv")
positions = load_positions_csv(SAMPLE_DATA_DIR / "positions.csv")
investor_classes = load_investor_classes_csv(SAMPLE_DATA_DIR / "investor_classes.csv")
redemption_scenarios = load_redemption_scenarios_csv(SAMPLE_DATA_DIR / "redemption_scenarios.csv")
market_stresses = load_market_stresses_csv(SAMPLE_DATA_DIR / "market_stresses.csv")
liquidity_stresses = load_liquidity_stresses_json(SAMPLE_DATA_DIR / "liquidity_stresses.json")
scenario_definitions = load_scenario_definitions_csv(SAMPLE_DATA_DIR / "scenario_definitions.csv")
lmt_parameters = load_lmt_parameters_csv(SAMPLE_DATA_DIR / "lmt_parameters.csv")
liquidation_strategies = load_liquidation_strategies_json(
    SAMPLE_DATA_DIR / "liquidation_strategies.json"
)

pd.DataFrame(
    [
        {"dataset": "funds", "records": len(funds)},
        {"dataset": "positions", "records": len(positions)},
        {"dataset": "investor_classes", "records": len(investor_classes)},
        {"dataset": "redemption_scenarios", "records": len(redemption_scenarios)},
        {"dataset": "market_stresses", "records": len(market_stresses)},
        {"dataset": "liquidity_stresses", "records": len(liquidity_stresses)},
        {"dataset": "scenario_definitions", "records": len(scenario_definitions)},
        {"dataset": "lmt_parameters", "records": len(lmt_parameters)},
        {"dataset": "liquidation_strategies", "records": len(liquidation_strategies)},
    ]
)

,dataset,records
0,funds,1
1,positions,9
2,investor_classes,5
3,redemption_scenarios,3
4,market_stresses,4
5,liquidity_stresses,3
6,scenario_definitions,6
7,lmt_parameters,2
8,liquidation_strategies,4


## Scenario Overview

This section shows the scenario definitions selected for inspection. It matters because each scenario maps a fund, redemption assumption, liquidity stress, liquidation strategy, and LMT parameter set into one engine run.

Interpret this table as the review menu: each row is one representative liquidation strategy run.

In [3]:
fund_by_key = {(fund.fund_id, fund.as_of_date): fund for fund in funds}
redemption_by_id = {scenario.redemption_scenario_id: scenario for scenario in redemption_scenarios}
market_stress_by_id = {stress.market_stress_id: stress for stress in market_stresses}
liquidity_stress_by_id = {stress.liquidity_stress_id: stress for stress in liquidity_stresses}
strategy_by_id = {strategy.liquidation_strategy_id: strategy for strategy in liquidation_strategies}
parameters_by_key = {
    (parameters.fund_id, parameters.as_of_date, parameters.parameter_set_id): parameters
    for parameters in lmt_parameters
}

pd.DataFrame(
    [
        {
            "scenario_id": scenario.scenario_id,
            "fund_id": scenario.fund_id,
            "as_of_date": str(scenario.as_of_date),
            "redemption_scenario_id": scenario.redemption_scenario_id,
            "market_stress_id": scenario.market_stress_id,
            "liquidity_stress_id": scenario.liquidity_stress_id,
            "liquidation_strategy_id": scenario.liquidation_strategy_id,
            "lmt_parameter_set_id": scenario.lmt_parameter_set_id,
            "strategy_type": strategy_by_id[scenario.liquidation_strategy_id].strategy_type.value,
        }
        for scenario in scenario_definitions
    ]
)

,scenario_id,fund_id,as_of_date,redemption_scenario_id,market_stress_id,liquidity_stress_id,liquidation_strategy_id,lmt_parameter_set_id,strategy_type
0,moderate_redemption_normal,lux_dynamic_allocation,2026-06-30,moderate_redemption_pressure,normal_market_conditions,normal_liquidity_capacity,cash_then_liquid_assets,board_approved_base,most_liquid_first
1,moderate_redemption_stress,lux_dynamic_allocation,2026-06-30,moderate_redemption_pressure,moderate_market_stress,normal_liquidity_capacity,cash_then_liquid_assets,board_approved_base,most_liquid_first
2,platform_outflow_moderate,lux_dynamic_allocation,2026-06-30,severe_platform_outflow,moderate_market_stress,reduced_equity_capacity,portfolio_profile_pro_rata,board_approved_base,pro_rata
3,platform_outflow_severe,lux_dynamic_allocation,2026-06-30,severe_platform_outflow,severe_market_stress,reduced_equity_capacity,portfolio_profile_pro_rata,board_approved_base,pro_rata
4,platform_outflow_hybrid,lux_dynamic_allocation,2026-06-30,severe_platform_outflow,severe_market_stress,reduced_equity_capacity,partial_cash_then_pro_rata,board_approved_base,hybrid
5,extreme_outflow_crisis,lux_dynamic_allocation,2026-06-30,severe_platform_outflow,historical_crisis_2008,reduced_equity_capacity,balanced_custom_weights,conservative_liquidity_buffer,custom_weights


## Local Diagnostic Helpers

The liquidation engine expects already-stressed liquidation positions and a redemption amount. This section prepares those inputs from the loaded sample data so the engine can be called.

This matters because the inspection reviews the liquidation strategy engine in isolation, rather than demonstrating the application service workflow. The helper uses documented sample assumptions for diagnostic review only: non-stress liquidity scenarios use zero haircut, stressed liquidity scenarios use positive haircut rates from the loaded position assumptions and liquidity stress multiplier. Market stress identifiers are shown for traceability, while the liquidation inspection keeps market values at the sample position values because market-stress methodology is outside this inspection scope.

In [4]:
SELLABLE_GROUPS = {
    AssetGroup.REVERSE_REPO,
    AssetGroup.LISTED_ETF,
    AssetGroup.LISTED_EQUITY,
}


def scenario_investors(scenario):
    return [
        investor
        for investor in investor_classes
        if investor.fund_id == scenario.fund_id and investor.as_of_date == scenario.as_of_date
    ]


def redemption_rows(scenario):
    fund = fund_by_key[(scenario.fund_id, scenario.as_of_date)]
    redemption = redemption_by_id[scenario.redemption_scenario_id]
    rows = []
    for investor in scenario_investors(scenario):
        redemption_amount = (
            fund.nav
            * investor.nav_share_rate
            * investor.stress_redemption_rate
            * redemption.redemption_multiplier
        )
        rows.append(
            {
                "scenario_id": scenario.scenario_id,
                "client_class": investor.client_class.value,
                "nav_share_rate": investor.nav_share_rate,
                "stress_redemption_rate": investor.stress_redemption_rate,
                "redemption_multiplier": redemption.redemption_multiplier,
                "redemption_amount": redemption_amount,
            }
        )
    return rows


def total_redemption_amount(scenario):
    return sum((row["redemption_amount"] for row in redemption_rows(scenario)), Decimal("0"))


def stressed_haircut_rate(position, liquidity_stress):
    if position.asset_group is AssetGroup.CASH:
        return Decimal("0")
    assumption = liquidity_stress.execution_assumptions_by_asset_group.get(position.asset_group)
    if assumption is None:
        return position.base_haircut_rate
    return min(position.base_haircut_rate + assumption.liquidity_haircut_rate, Decimal("1"))


def stressed_liquidity_capacity_rate(position, liquidity_stress):
    if position.asset_group is AssetGroup.CASH:
        return Decimal("1")
    assumption = liquidity_stress.execution_assumptions_by_asset_group.get(position.asset_group)
    if assumption is None:
        return position.base_liquidity_capacity_rate
    return position.base_liquidity_capacity_rate * assumption.participation_rate


def shocked_market_value(position, market_stress):
    if position.market_value is None:
        return Decimal("0")
    if position.asset_group not in {AssetGroup.LISTED_ETF, AssetGroup.LISTED_EQUITY}:
        return position.market_value
    return max(
        position.market_value * (Decimal("1") + market_stress.market_shock_rate),
        Decimal("0"),
    )


def scenario_positions(scenario):
    liquidity_stress = liquidity_stress_by_id[scenario.liquidity_stress_id]
    market_stress = market_stress_by_id[scenario.market_stress_id]
    return [
        StressedLiquidationPosition(
            position_id=position.position_id,
            asset_group=position.asset_group,
            stressed_market_value=shocked_market_value(position, market_stress),
            stressed_haircut_rate=stressed_haircut_rate(position, liquidity_stress),
            stressed_liquidity_capacity_rate=stressed_liquidity_capacity_rate(
                position, liquidity_stress
            ),
            settlement_days=position.settlement_days,
            maturity_days=position.maturity_days,
            notional_amount=position.notional_amount,
        )
        for position in positions
        if position.fund_id == scenario.fund_id and position.as_of_date == scenario.as_of_date
    ]


def is_eligible(position, stress_horizon_days):
    if position.asset_group not in SELLABLE_GROUPS:
        return False
    if position.stressed_market_value is None or position.stressed_market_value <= Decimal("0"):
        return False
    if position.asset_group is AssetGroup.REVERSE_REPO:
        if position.maturity_days is None:
            return False
        return position.maturity_days + position.settlement_days <= stress_horizon_days
    return position.settlement_days <= stress_horizon_days


def run_scenario(scenario):
    fund = fund_by_key[(scenario.fund_id, scenario.as_of_date)]
    liquidity_stress = liquidity_stress_by_id[scenario.liquidity_stress_id]
    strategy = strategy_by_id[scenario.liquidation_strategy_id]
    parameters = parameters_by_key[
        (scenario.fund_id, scenario.as_of_date, scenario.lmt_parameter_set_id)
    ]
    redemption_amount = total_redemption_amount(scenario)
    liquidation_positions = scenario_positions(scenario)
    result = calculate_liquidation_strategy(
        scenario_id=scenario.scenario_id,
        fund=fund,
        positions=liquidation_positions,
        redemption_amount=redemption_amount,
        strategy=strategy,
        lmt_parameters=parameters,
        stress_horizon_days=liquidity_stress.stress_horizon_days,
    )
    return {
        "scenario": scenario,
        "fund": fund,
        "liquidity_stress": liquidity_stress,
        "market_stress": market_stress_by_id[scenario.market_stress_id],
        "strategy": strategy,
        "parameters": parameters,
        "redemption_amount": redemption_amount,
        "positions": liquidation_positions,
        "result": result,
    }


runs = [run_scenario(scenario) for scenario in scenario_definitions]

## Scenario Inputs

This section summarizes the key inputs passed into each liquidation run: NAV, total redemption amount, selected strategy, liquidity horizon, and configured LMT buffer.

Interpret `total_redemption_rate` as the liability-side pressure being tested. The liquidation engine then tries to raise this amount while respecting the strategy, haircuts, capacity, settlement, maturity, and cash-buffer rules.

In [5]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "market_stress_id": run["market_stress"].market_stress_id,
            "liquidity_stress_id": run["liquidity_stress"].liquidity_stress_id,
            "stress_horizon_days": run["liquidity_stress"].stress_horizon_days,
            "nav": money(run["fund"].nav),
            "total_redemption_amount": money(run["redemption_amount"]),
            "total_redemption_rate": rate(run["redemption_amount"] / run["fund"].nav),
            "minimum_buffer_rate": rate(run["parameters"].minimum_buffer_rate),
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,market_stress_id,liquidity_stress_id,stress_horizon_days,nav,total_redemption_amount,total_redemption_rate,minimum_buffer_rate
0,moderate_redemption_normal,most_liquid_first,normal_market_conditions,normal_liquidity_capacity,5,"100,000,000.00","11,500,000.00",11.50%,5.00%
1,moderate_redemption_stress,most_liquid_first,moderate_market_stress,normal_liquidity_capacity,5,"100,000,000.00","11,500,000.00",11.50%,5.00%
2,platform_outflow_moderate,pro_rata,moderate_market_stress,reduced_equity_capacity,5,"100,000,000.00","17,250,000.00",17.25%,5.00%
3,platform_outflow_severe,pro_rata,severe_market_stress,reduced_equity_capacity,5,"100,000,000.00","17,250,000.00",17.25%,5.00%
4,platform_outflow_hybrid,hybrid,severe_market_stress,reduced_equity_capacity,5,"100,000,000.00","17,250,000.00",17.25%,5.00%
5,extreme_outflow_crisis,custom_weights,historical_crisis_2008,reduced_equity_capacity,5,"100,000,000.00","17,250,000.00",17.25%,8.00%


## Redemption Amount By Investor Class

This section shows how the sample investor-class assumptions contribute to each scenario's redemption amount. It matters because the liquidation engine receives a single redemption amount, but the review should still see where that amount came from.

Interpret larger class-level rows as the investor groups driving cash demand in the liquidation run.

In [6]:
redemption_table = pd.DataFrame(
    {
        **row,
        "nav_share_rate": rate(row["nav_share_rate"]),
        "stress_redemption_rate": rate(row["stress_redemption_rate"]),
        "redemption_multiplier": str(row["redemption_multiplier"]),
        "redemption_amount": money(row["redemption_amount"]),
    }
    for scenario in scenario_definitions
    for row in redemption_rows(scenario)
)
redemption_table

,scenario_id,client_class,nav_share_rate,stress_redemption_rate,redemption_multiplier,redemption_amount
0,moderate_redemption_normal,retail,30.00%,8.00%,1.00,"2,400,000.00"
1,moderate_redemption_normal,institutional,25.00%,12.00%,1.00,"3,000,000.00"
2,moderate_redemption_normal,platform,25.00%,18.00%,1.00,"4,500,000.00"
3,moderate_redemption_normal,fund_of_funds,15.00%,10.00%,1.00,"1,500,000.00"
4,moderate_redemption_normal,seed_capital,5.00%,2.00%,1.00,"100,000.00"
5,moderate_redemption_stress,retail,30.00%,8.00%,1.00,"2,400,000.00"
6,moderate_redemption_stress,institutional,25.00%,12.00%,1.00,"3,000,000.00"
7,moderate_redemption_stress,platform,25.00%,18.00%,1.00,"4,500,000.00"
8,moderate_redemption_stress,fund_of_funds,15.00%,10.00%,1.00,"1,500,000.00"
9,moderate_redemption_stress,seed_capital,5.00%,2.00%,1.00,"100,000.00"


## Cash Buffer Calculation

This section shows how much cash is available above the configured minimum buffer. It matters because V1 strategies preserve the minimum buffer and should not automatically drain all cash.

Interpret `cash_above_buffer` as the maximum cash that a cash-using strategy can consume before selling eligible non-cash assets.

In [7]:
cash_buffer_rows = []
for run in runs:
    cash_total = sum(
        (
            position.stressed_market_value or Decimal("0")
            for position in run["positions"]
            if position.asset_group is AssetGroup.CASH
        ),
        Decimal("0"),
    )
    minimum_cash_buffer = run["fund"].nav * run["parameters"].minimum_buffer_rate
    cash_buffer_rows.append(
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "cash_total": money(cash_total),
            "minimum_cash_buffer": money(minimum_cash_buffer),
            "cash_above_buffer": money(max(cash_total - minimum_cash_buffer, Decimal("0"))),
            "cash_used": money(run["result"].cash_used),
            "minimum_cash_buffer_preserved": run["result"].minimum_cash_buffer_preserved,
        }
    )

pd.DataFrame(cash_buffer_rows)

,scenario_id,strategy_type,cash_total,minimum_cash_buffer,cash_above_buffer,cash_used,minimum_cash_buffer_preserved
0,moderate_redemption_normal,most_liquid_first,"12,000,000.00","5,000,000.00","7,000,000.00","7,000,000.00",True
1,moderate_redemption_stress,most_liquid_first,"12,000,000.00","5,000,000.00","7,000,000.00","7,000,000.00",True
2,platform_outflow_moderate,pro_rata,"12,000,000.00","5,000,000.00","7,000,000.00",0.00,True
3,platform_outflow_severe,pro_rata,"12,000,000.00","5,000,000.00","7,000,000.00",0.00,True
4,platform_outflow_hybrid,hybrid,"12,000,000.00","5,000,000.00","7,000,000.00","3,500,000.00",True
5,extreme_outflow_crisis,custom_weights,"12,000,000.00","8,000,000.00","4,000,000.00",0.00,True


## Eligible Assets

This section shows which positions can provide usable liquidity within each scenario's stress horizon. It matters because the engine excludes repo financing from ordinary liquidation and applies settlement-day and reverse-repo maturity rules before allocating sales.

Interpret `eligible_for_liquidation` as whether the asset can be sold or matured within the horizon for this one-period run. Cash appears here for context but is not treated as an ordinary sellable asset.

In [8]:
eligible_rows = []
for run in runs:
    horizon = run["liquidity_stress"].stress_horizon_days
    for position in run["positions"]:
        available_capacity = None
        if position.stressed_market_value is not None:
            available_capacity = (
                position.stressed_market_value * position.stressed_liquidity_capacity_rate
            )
        eligible_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "position_id": position.position_id,
                "asset_group": position.asset_group.value,
                "stressed_market_value": money(position.stressed_market_value),
                "stressed_haircut_rate": rate(position.stressed_haircut_rate),
                "stressed_liquidity_capacity_rate": rate(position.stressed_liquidity_capacity_rate),
                "available_capacity": money(available_capacity),
                "settlement_days": days(position.settlement_days),
                "maturity_days": days(position.maturity_days),
                "stress_horizon_days": horizon,
                "eligible_for_liquidation": is_eligible(position, horizon),
            }
        )

pd.DataFrame(eligible_rows)

,scenario_id,position_id,asset_group,stressed_market_value,stressed_haircut_rate,stressed_liquidity_capacity_rate,available_capacity,settlement_days,maturity_days,stress_horizon_days,eligible_for_liquidation
0,moderate_redemption_normal,eur_operating_cash,cash,"12,000,000.00",0.00%,100.00%,"12,000,000.00",0,,5,False
1,moderate_redemption_normal,sap_equity_position,listed_equity,"15,000,000.00",10.00%,6.00%,"900,000.00",2,,5,True
2,moderate_redemption_normal,asml_equity_position,listed_equity,"12,000,000.00",11.00%,5.40%,"648,000.00",2,,5,True
3,moderate_redemption_normal,lvmh_equity_position,listed_equity,"8,000,000.00",11.00%,5.40%,"432,000.00",2,,5,True
4,moderate_redemption_normal,msci_world_etf_position,listed_etf,"18,000,000.00",6.00%,14.00%,"2,520,000.00",2,,5,True
5,moderate_redemption_normal,euro_stoxx_etf_position,listed_etf,"10,000,000.00",6.00%,16.00%,"1,600,000.00",2,,5,True
6,moderate_redemption_normal,overnight_reverse_repo_bnp,reverse_repo,"15,000,000.00",1.00%,100.00%,"15,000,000.00",1,1,5,True
7,moderate_redemption_normal,one_week_reverse_repo_sg,reverse_repo,"10,000,000.00",1.00%,90.00%,"9,000,000.00",1,7,5,False
8,moderate_redemption_normal,eur_repo_financing_obligation,repo_financing,0.00,0.00%,0.00%,0.00,1,,5,False
9,moderate_redemption_stress,eur_operating_cash,cash,"12,000,000.00",0.00%,100.00%,"12,000,000.00",0,,5,False


## Liquidation Strategy Execution

This section summarizes the engine output for each scenario after calling `calculate_liquidation_strategy`. It matters because this is the central review point: the engine must convert redemption need and stressed liquidity assumptions into cash raised, dilution, shortfall, and remaining buffer metrics.

Interpret `shortfall` as unmet redemption need after cash and post-haircut asset sales. Interpret `remaining_liquid_buffer_rate` as post-haircut eligible liquidity plus remaining cash divided by NAV after redemption and realised liquidation cost, before simulated LMT effects.

In [9]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "total_redemption_amount": money(run["result"].total_redemption_amount),
            "cash_used": money(run["result"].cash_used),
            "post_haircut_cash_raised": money(run["result"].total_post_haircut_cash_raised),
            "dilution_amount": money(run["result"].dilution_amount),
            "dilution_rate": rate(run["result"].dilution_rate),
            "shortfall": money(run["result"].shortfall),
            "remaining_liquid_buffer_rate": rate(run["result"].remaining_liquid_buffer_rate),
            "minimum_cash_buffer_preserved": run["result"].minimum_cash_buffer_preserved,
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,total_redemption_amount,cash_used,post_haircut_cash_raised,dilution_amount,dilution_rate,shortfall,remaining_liquid_buffer_rate,minimum_cash_buffer_preserved
0,moderate_redemption_normal,most_liquid_first,"11,500,000.00","7,000,000.00","4,500,000.00","45,454.55",0.05%,0.00,23.73%,True
1,moderate_redemption_stress,most_liquid_first,"11,500,000.00","7,000,000.00","4,500,000.00","45,454.55",0.05%,0.00,24.28%,True
2,platform_outflow_moderate,pro_rata,"17,250,000.00",0.00,"5,887,203.83","574,124.86",0.57%,"11,362,796.17",25.54%,True
3,platform_outflow_severe,pro_rata,"17,250,000.00",0.00,"5,924,555.01","546,392.42",0.55%,"11,325,444.99",26.75%,True
4,platform_outflow_hybrid,hybrid,"17,250,000.00","3,500,000.00","5,179,239.85","523,341.44",0.52%,"8,570,760.15",23.05%,True
5,extreme_outflow_crisis,custom_weights,"17,250,000.00",0.00,"5,112,830.00","426,371.03",0.43%,"12,137,170.00",33.50%,True


## Liquidation Allocation

This section shows the position-level gross sale allocation selected by each strategy. It matters because the strategy choice should be visible in which assets are used and in what amount.

Interpret `gross_sale_amount` as the amount sold before haircut. For positive haircut scenarios, gross sales can be higher than the post-haircut cash raised.

In [10]:
allocation_rows = []
for run in runs:
    for asset in run["result"].assets_liquidated:
        allocation_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "strategy_type": run["strategy"].strategy_type.value,
                "position_id": asset.position_id,
                "asset_group": asset.asset_group.value,
                "gross_sale_amount": money(asset.gross_sale_amount),
            }
        )

pd.DataFrame(allocation_rows)

,scenario_id,strategy_type,position_id,asset_group,gross_sale_amount
0,moderate_redemption_normal,most_liquid_first,overnight_reverse_repo_bnp,reverse_repo,"4,545,454.55"
1,moderate_redemption_stress,most_liquid_first,overnight_reverse_repo_bnp,reverse_repo,"4,545,454.55"
2,platform_outflow_moderate,pro_rata,asml_equity_position,listed_equity,"307,800.00"
3,platform_outflow_moderate,pro_rata,euro_stoxx_etf_position,listed_etf,"760,000.00"
4,platform_outflow_moderate,pro_rata,lvmh_equity_position,listed_equity,"205,200.00"
5,platform_outflow_moderate,pro_rata,msci_world_etf_position,listed_etf,"1,197,000.00"
6,platform_outflow_moderate,pro_rata,overnight_reverse_repo_bnp,reverse_repo,"3,563,828.69"
7,platform_outflow_moderate,pro_rata,sap_equity_position,listed_equity,"427,500.00"
8,platform_outflow_severe,pro_rata,asml_equity_position,listed_equity,"285,120.00"
9,platform_outflow_severe,pro_rata,euro_stoxx_etf_position,listed_etf,"704,000.00"


## Post-Haircut Cash Raised

This section shows the cash generated after applying scenario-driven haircut rates to gross sales. It matters because the engine targets post-haircut cash need when capacity allows, rather than treating gross sale amount as usable cash.

Interpret the difference between `gross_sale_amount` and `post_haircut_cash_raised` as haircut leakage that contributes to dilution cost.

In [11]:
post_haircut_rows = []
for run in runs:
    for asset in run["result"].assets_liquidated:
        post_haircut_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "position_id": asset.position_id,
                "asset_group": asset.asset_group.value,
                "gross_sale_amount": money(asset.gross_sale_amount),
                "post_haircut_cash_raised": money(asset.post_haircut_cash_raised),
                "haircut_cost": money(asset.haircut_cost),
            }
        )

pd.DataFrame(post_haircut_rows)

,scenario_id,position_id,asset_group,gross_sale_amount,post_haircut_cash_raised,haircut_cost
0,moderate_redemption_normal,overnight_reverse_repo_bnp,reverse_repo,"4,545,454.55","4,500,000.00","45,454.55"
1,moderate_redemption_stress,overnight_reverse_repo_bnp,reverse_repo,"4,545,454.55","4,500,000.00","45,454.55"
2,platform_outflow_moderate,asml_equity_position,listed_equity,"307,800.00","243,162.00","64,638.00"
3,platform_outflow_moderate,euro_stoxx_etf_position,listed_etf,"760,000.00","653,600.00","106,400.00"
4,platform_outflow_moderate,lvmh_equity_position,listed_equity,"205,200.00","162,108.00","43,092.00"
5,platform_outflow_moderate,msci_world_etf_position,listed_etf,"1,197,000.00","1,029,420.00","167,580.00"
6,platform_outflow_moderate,overnight_reverse_repo_bnp,reverse_repo,"3,563,828.69","3,456,913.83","106,914.86"
7,platform_outflow_moderate,sap_equity_position,listed_equity,"427,500.00","342,000.00","85,500.00"
8,platform_outflow_severe,asml_equity_position,listed_equity,"285,120.00","225,244.80","59,875.20"
9,platform_outflow_severe,euro_stoxx_etf_position,listed_etf,"704,000.00","605,440.00","98,560.00"


## Dilution Cost

This section aggregates haircut cost into the engine's dilution estimate. Dilution remains important calibration context, while the simulated swing activation assessment compares the redemption rate with the swing activation threshold.

Interpret higher `dilution_rate` as more liquidation cost relative to NAV. The inspection does not decide whether an LMT should be activated.

In [12]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "dilution_amount": money(run["result"].dilution_amount),
            "dilution_rate": rate(run["result"].dilution_rate),
            "swing_threshold_rate": rate(run["parameters"].swing_threshold_rate),
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,dilution_amount,dilution_rate,swing_threshold_rate
0,moderate_redemption_normal,most_liquid_first,"45,454.55",0.05%,1.50%
1,moderate_redemption_stress,most_liquid_first,"45,454.55",0.05%,1.50%
2,platform_outflow_moderate,pro_rata,"574,124.86",0.57%,1.50%
3,platform_outflow_severe,pro_rata,"546,392.42",0.55%,1.50%
4,platform_outflow_hybrid,hybrid,"523,341.44",0.52%,1.50%
5,extreme_outflow_crisis,custom_weights,"426,371.03",0.43%,1.00%


## Shortfall

This section isolates remaining unmet redemption need after cash use and post-haircut liquidation proceeds. It matters because shortfall indicates that eligible liquidity, after capacity and haircut constraints, was insufficient for the scenario.

Interpret zero shortfall as the strategy raising enough cash within the one-period stress horizon. Positive shortfall should be reviewed as a methodology or parameter concern in a separate ticket, not changed inside this inspection.

In [13]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "total_redemption_amount": money(run["result"].total_redemption_amount),
            "total_cash_raised": money(
                run["result"].cash_used + run["result"].total_post_haircut_cash_raised
            ),
            "shortfall": money(run["result"].shortfall),
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,total_redemption_amount,total_cash_raised,shortfall
0,moderate_redemption_normal,most_liquid_first,"11,500,000.00","11,500,000.00",0.00
1,moderate_redemption_stress,most_liquid_first,"11,500,000.00","11,500,000.00",0.00
2,platform_outflow_moderate,pro_rata,"17,250,000.00","5,887,203.83","11,362,796.17"
3,platform_outflow_severe,pro_rata,"17,250,000.00","5,924,555.01","11,325,444.99"
4,platform_outflow_hybrid,hybrid,"17,250,000.00","8,679,239.85","8,570,760.15"
5,extreme_outflow_crisis,custom_weights,"17,250,000.00","5,112,830.00","12,137,170.00"


## Allocation By Asset Group

This section groups gross liquidation allocations by asset group. It matters because reviewers can compare strategy behavior at a portfolio level without reading every position row.

Interpret these values as gross allocation amounts, not post-haircut cash raised. Use the post-haircut section to inspect usable cash.

In [14]:
asset_group_rows = []
for run in runs:
    for asset_group, allocation in run["result"].asset_group_allocations.items():
        asset_group_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "strategy_type": run["strategy"].strategy_type.value,
                "asset_group": asset_group.value,
                "gross_allocation_amount": money(allocation),
            }
        )

pd.DataFrame(asset_group_rows).sort_values(["scenario_id", "asset_group"])

,scenario_id,strategy_type,asset_group,gross_allocation_amount
16,extreme_outflow_crisis,custom_weights,listed_equity,"643,500.00"
15,extreme_outflow_crisis,custom_weights,listed_etf,"1,339,000.00"
14,extreme_outflow_crisis,custom_weights,reverse_repo,"3,556,701.03"
0,moderate_redemption_normal,most_liquid_first,cash,"7,000,000.00"
1,moderate_redemption_normal,most_liquid_first,reverse_repo,"4,545,454.55"
2,moderate_redemption_stress,most_liquid_first,cash,"7,000,000.00"
3,moderate_redemption_stress,most_liquid_first,reverse_repo,"4,545,454.55"
10,platform_outflow_hybrid,hybrid,cash,"3,500,000.00"
11,platform_outflow_hybrid,hybrid,listed_equity,"871,200.00"
12,platform_outflow_hybrid,hybrid,listed_etf,"1,812,800.00"
